## 1. 라이브러리 로드

In [105]:
# 라이브러리 호출
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import platform
import ast
from collections import Counter
import json

warnings.filterwarnings('ignore')

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # Mac
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 컬럼 너비 제한 해제
pd.set_option('display.max_colwidth', None)

---
## 2. 데이터 로드

In [106]:
df_platinum_Match = pd.read_csv('TFT_Platinum_MatchData.csv')
df_diamond_Match = pd.read_csv('TFT_Diamond_MatchData.csv')
df_master_Match = pd.read_csv('TFT_Master_MatchData.csv')
df_grand_master_Match = pd.read_csv('TFT_GrandMaster_MatchData.csv')
df_challenger_Match = pd.read_csv('TFT_Challenger_MatchData.csv')

df_champion_info = pd.read_csv('TFT_Champion_CurrentVersion.csv')
df_items_info = pd.read_csv('TFT_Item_CurrentVersion.csv')


In [107]:
# 매치관련 데이터가 담긴 딕셔너리 정의
match_data = {
    'platinum': df_platinum_Match,
    'diamond': df_diamond_Match,
    'master': df_master_Match,
    'grand_master': df_grand_master_Match,
    'challenger': df_challenger_Match,
}

# 각 티어별 테이블에 'Tier'정보가 담긴 컬럼 추가
for name, df in match_data.items():
    df['tier'] = name


# 모든 티어의 경기데이터가 담긴 데이터프레임 제작
# ignore_index=True: 데이터를 병합한 뒤, 새로운 인덱스 부여
df_all_match = pd.concat(match_data.values(), ignore_index=True)

df_all_match.head(3)

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion,tier
0,KR_4291707834,1963.905273,6,27,5,1390.165771,"{'Cybernetic': 1, 'Demolitionist': 1, 'Infiltrator': 1, 'Rebel': 1, 'Set3_Brawler': 1, 'Set3_Celestial': 1, 'Set3_Void': 1, 'Sniper': 1}","{'Ziggs': {'items': [7], 'star': 1}, 'Ashe': {'items': [9], 'star': 1}, 'ChoGath': {'items': [6], 'star': 1}, 'Ekko': {'items': [1], 'star': 1}}",platinum
1,KR_4291707834,1963.905273,8,37,3,1891.282715,"{'Blaster': 1, 'Chrono': 1, 'Cybernetic': 4, 'Demolitionist': 1, 'Rebel': 1, 'Set3_Blademaster': 2, 'Set3_Brawler': 1, 'Set3_Sorcerer': 1, 'Set3_Void': 1, 'Valkyrie': 1, 'Vanguard': 2}","{'Ziggs': {'items': [24], 'star': 3}, 'Fiora': {'items': [37], 'star': 2}, 'Leona': {'items': [36, 24], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [5], 'star': 2}, 'Kayle': {'items': [], 'star': 2}, 'WuKong': {'items': [3, 67], 'star': 2}, 'VelKoz': {'items': [4], 'star': 2}}",platinum
2,KR_4291707834,1963.905273,6,25,7,1279.461060,"{'Blaster': 1, 'Cybernetic': 1, 'DarkStar': 2, 'Infiltrator': 1, 'Mercenary': 1, 'Set3_Blademaster': 1, 'Set3_Mystic': 1, 'Valkyrie': 1}","{'Fiora': {'items': [1], 'star': 1}, 'Shaco': {'items': [6], 'star': 1}, 'Karma': {'items': [4], 'star': 1}, 'MissFortune': {'items': [3], 'star': 1}}",platinum


---
## 3. 데이터 전처리
### 3-1. 중복행 제거

In [108]:
# 중복행 제거
duplicates = df_all_match[df_all_match.duplicated(keep=False)]

print(f"중복 제거 전: {len(df_all_match)}")
print(f"중복된 행 개수: {len(duplicates)}")

# 결과 확인
df_all_match = df_all_match.drop_duplicates()
print(f"\n중복 제거 후: {len(df_all_match)}")

중복 제거 전: 399998
중복된 행 개수: 80

중복 제거 후: 399930


### 3-2. 컬럼명 소문자로 통일

In [109]:
# 전체 컬럼명 소문자 통일
df_all_match.columns = df_all_match.columns.str.lower()

print(df_all_match.columns)

Index(['gameid', 'gameduration', 'level', 'lastround', 'ranked',
       'ingameduration', 'combination', 'champion', 'tier'],
      dtype='str')


### 3-3. ranked=0인 데이터 삭제

In [110]:
# ranked = 0이 포함된 경기 데이터 삭제
df_all_match_2 = df_all_match.copy()

# ranked==0인 행의 gameId 추출
drop_game_ids = df_all_match_2[df_all_match_2['ranked']==0]['gameid'].unique()

# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx)

print(f"ranked = 0인 데이터가 포함된 경기 데이터 삭제하기 전: {df_all_match.shape}")
print(f"ranked = 0인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

ranked = 0인 데이터가 포함된 경기 데이터 삭제하기 전: (399930, 9)
ranked = 0인 데이터가 포함된 경기 데이터 삭제한 후: (399886, 9)


### 3-4. 경기시간 = 0인 데이터 삭제

In [111]:
# 전체 게임시간 = 0인 행의 GameId 추출
zero_duration_ids = set(df_all_match_2[df_all_match_2['gameduration'] == 0]['gameid'])

# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_2 = df_all_match_2[df_all_match_2['gameid'].isin(zero_duration_ids)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_2)

print(f"gameduration = 0인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

gameduration = 0인 데이터가 포함된 경기 데이터 삭제한 후: (399886, 9)


### 3-5. 경기 참여 유저 수가 8명 미만인 게임 삭제

In [112]:
# 게임 ID별 플레이어 수 정보 추가
df_all_match_2['player_cnt'] = df_all_match_2['gameid'].map(df_all_match_2['gameid'].value_counts())

# player_cnt < 8인 행의 gameId 추출
drop_game_ids_3 = df_all_match_2[df_all_match_2['player_cnt'] < 8]['gameid'].unique()

# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_3 = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids_3)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_3)

print(f"player_cnt < 8인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

player_cnt < 8인 데이터가 포함된 경기 데이터 삭제한 후: (399872, 10)


### 3-6. 시즌2 데이터 삭제

In [113]:
# 시즌 2 고유 시너지 목록
season2_keys = {'Alchemist', 'Avatar', 'Berserker',
                'Crystal', 'Desert', 'Druid', 'Electric',
                'Light', 'Mage', 'Mountain', 'Mystic', 'Ocean',
                'Poison', 'Predator', 'Set2_Assassin', 'Set2_Blademaster',
                'Set2_Glacial', 'Set2_Ranger', 'Shadow', 'Soulbound',
                'Summoner', 'Warden', 'Wind', 'Woodland'}  # 시즌 2 시너지 입력


# 시즌 2 시너지가 하나라도 포함되면 시즌 2로 분류
df_all_match_2['season'] = df_all_match_2['combination'].apply(
    lambda x: 'season 2' if any(k in season2_keys for k in ast.literal_eval(x).keys())
    else 'season 3'
)

# season 2인 행의 gameId 추출
drop_game_ids_4 = df_all_match_2[df_all_match_2['season'] == 'season 2']['gameid'].unique()


# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_4 = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids_4)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_4)

print(f"season 2인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

season 2인 데이터가 포함된 경기 데이터 삭제한 후: (396640, 11)


### 3-7. 시즌 3 시너지 더미 데이터 ”TemplateTrait” 삭제
- 시너지 데이터 중 해당 값인 `{'TemplateTrait': 1}`만 삭제

In [114]:
# TemplateTrait 키가 있는 행만 필터링 후 값 확인
df_all_match_2[df_all_match_2['combination'].apply(
    lambda x: 'TemplateTrait' in ast.literal_eval(x).keys() # TemplateTrait 키가 있는 행만 필터링
)]['combination'].apply(                                    # 필터링된 행 중에서 combination 컬럼만 가져옴
    lambda x: ast.literal_eval(x).get('TemplateTrait')      # TemplateTrait 키에 할당된 값을 가져옴
).value_counts()

combination
1    65987
Name: count, dtype: int64

In [115]:
# 분석 목적에 영향을 주지 않는 dummy 데이터로 판단하여 시너지 중 'TemplateTrait'만 삭제
df_all_match_2['combination'] = df_all_match_2['combination'].apply(
    lambda x: json.dumps(
        # key: value = 키-값의 쌍을 의미함
        {key: value for key, value in ast.literal_eval(x).items() if key != 'TemplateTrait'},
        ensure_ascii=False
    )
)

In [116]:
# TemplateTrait 키가 있는 행 수 확인 → (0,11)이면 완전히 삭제된 것
df_all_match_2[df_all_match_2['combination'].apply(
    lambda x: 'TemplateTrait' in ast.literal_eval(x).keys()
)].shape

(0, 11)

### 3-8. 유저 ID 컬럼 제작

In [117]:
df_all_match_2['user_id'] = 'KR-USER-' + (df_all_match_2.index + 1).astype(str)

In [118]:
df_all_match_2['user_id']

0              KR-USER-1
1              KR-USER-2
2              KR-USER-3
3              KR-USER-4
4              KR-USER-5
               ...      
399993    KR-USER-399994
399994    KR-USER-399995
399995    KR-USER-399996
399996    KR-USER-399997
399997    KR-USER-399998
Name: user_id, Length: 396640, dtype: str

### 3-9. 티어가 섞인 경기 정보 추출

In [119]:
# 티어가 섞인 gameId 추출
mixed_gameids = df_all_match_2.groupby('gameid')['tier'].nunique() # gameid별로 tier의 고유값 개수를 계산
mixed_gameids = mixed_gameids[mixed_gameids > 1].index # 고유값이 2 이상인 gameId만 추출

# 해당 gameId의 티어 구성 확인
# mixed_gameids에 포함된 gameId를 가진 행만 추출 -> gameid별로 어떤 티어가 몇 명씩 있는지 카운트
df_all_match_2[df_all_match_2['gameid'].isin(mixed_gameids)].groupby('gameid')['tier'].value_counts()

gameid         tier        
KR_4263820118  platinum        8
               master          8
KR_4313697214  master          8
               platinum        8
KR_4320079059  platinum        8
               diamond         8
KR_4344513862  master          8
               diamond         8
KR_4347884427  diamond         8
               master          8
KR_4357966612  grand_master    8
               platinum        8
KR_4358922415  master          8
               diamond         8
KR_4361594426  master          8
               diamond         8
KR_4361773461  diamond         8
               grand_master    8
KR_4362846604  master          8
               diamond         8
KR_4364453473  grand_master    8
               diamond         8
KR_4365284161  master          8
               diamond         8
KR_4378896137  diamond         8
               platinum        8
KR_4381231118  diamond         8
               platinum        8
KR_4387025966  diamond         8
               

In [120]:
df_all_match_2.groupby(['gameid', 'tier']).size().value_counts()

8    49580
Name: count, dtype: int64

# combination & champion 결측 처리 시작

### 1-1 combination & champion = {} 행 제거

In [121]:
#둘 다 {} 이면 행 제거
df_all_match_2 = df_all_match_2[
    ~((df_all_match_2['combination'] == '{}') & 
      (df_all_match_2['champion'] == '{}'))
].reset_index(drop=True)

### 1-2 combination O + champion X = flag_1 불리언 파생 컬럼 생성

In [122]:
#flag_1: combination O + champion X
df_all_match_2['flag_1'] = (
    (df_all_match_2['combination'] != '{}') & 
    (df_all_match_2['champion'] == '{}')
).astype(int)
print(f"flag_1 개수: {df_all_match_2['flag_1'].sum()}")

flag_1 개수: 35


In [123]:
df_all_match_2['flag_1'].value_counts()

flag_1
0    396342
1        35
Name: count, dtype: int64

### 1-3 combination X + champion O = flag_2 불리언 파생 컬럼 생성

In [124]:
#flag_2: combination X + champion O
df_all_match_2['flag_2'] = (
    (df_all_match_2['combination'] == '{}') & 
    (df_all_match_2['champion'] != '{}')
).astype(int)
print(f"flag_2 개수: {df_all_match_2['flag_2'].sum()}")

flag_2 개수: 117


In [125]:
df_all_match_2['champion']

0                                                                                                                                                                                                                                                {'Ziggs': {'items': [7], 'star': 1}, 'Ashe': {'items': [9], 'star': 1}, 'ChoGath': {'items': [6], 'star': 1}, 'Ekko': {'items': [1], 'star': 1}}
1                                                                                       {'Ziggs': {'items': [24], 'star': 3}, 'Fiora': {'items': [37], 'star': 2}, 'Leona': {'items': [36, 24], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [5], 'star': 2}, 'Kayle': {'items': [], 'star': 2}, 'WuKong': {'items': [3, 67], 'star': 2}, 'VelKoz': {'items': [4], 'star': 2}}
2                                                                                                                                                                                                                                   

In [126]:
df_champion_info.head()

,name,cost,health,defense,attack,attack_range,speed_of_attack,dps,skill_name,skill_cost,origin,class
0,gangplank,5,1000,30,60,1,1.00,60,gangplank_orbitalstrike,100/175,Space Pirate,"['Mercenary', 'Demolitionist']"
1,graves,1,650,35,55,1,0.55,30,graves_smokegrenade,50/80,Space Pirate,['Blaster']
2,neeko,3,800,35,50,2,0.65,33,neeko_popblossom,75/150,Star Guardian,['Protector']
3,darius,2,750,35,60,1,0.65,39,darius_spacepirateguillotine,0/60,Space Pirate,['Mana-Reaver']
4,rakan,2,600,35,45,2,0.70,32,rakan_grandentrance,50/100,Celestial,['Protector']


In [127]:
#str -> dic로 변경
import ast

df_all_match_2['combination'] = df_all_match_2['combination'].apply(ast.literal_eval)
df_all_match_2['champion'] = df_all_match_2['champion'].apply(ast.literal_eval)
print(type(df_all_match_2.loc[0, 'combination']))
print(type(df_all_match_2.loc[0, 'champion']))

<class 'dict'>
<class 'dict'>


In [128]:
mask = (
    (df_all_match_2['champion'] != {}) &
    (df_all_match_2['combination'] == {})
)

df_all_match_2[mask].head()

,gameid,gameduration,level,lastround,ranked,ingameduration,combination,champion,tier,player_cnt,season,user_id,flag_1,flag_2
6522,KR_4271274609,2037.501343,6,13,8,637.688232,{},"{'Fiora': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 2}, 'KaiSa': {'items': [], 'star': 1}, 'Mordekaiser': {'items': [], 'star': 2}, 'Kassadin': {'items': [], 'star': 1}}",platinum,8,season 3,KR-USER-6628,0,1
9873,KR_4359182660,2064.300293,5,13,8,621.877319,{},"{'JarvanIV': {'items': [], 'star': 1}, 'Lux': {'items': [], 'star': 1}}",platinum,8,season 3,KR-USER-10006,0,1
15744,KR_4320179268,2129.208252,4,12,8,604.033447,{},"{'Malphite': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 1}, 'Mordekaiser': {'items': [], 'star': 1}, 'Ashe': {'items': [], 'star': 1}}",platinum,8,season 3,KR-USER-15898,0,1
20393,KR_4380989360,2041.039062,8,37,2,2032.800903,{},"{'Fiora': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [], 'star': 2}, 'Irelia': {'items': [16, 19, 29], 'star': 2}, 'WuKong': {'items': [], 'star': 2}, 'Thresh': {'items': [], 'star': 1}, 'Ekko': {'items': [], 'star': 2}}",platinum,8,season 3,KR-USER-20910,0,1
21498,KR_4366152411,1980.549683,7,24,8,1259.300781,{},"{'Malphite': {'items': [], 'star': 2}, 'KhaZix': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 3}, 'Mordekaiser': {'items': [], 'star': 2}, 'Shaco': {'items': [], 'star': 2}, 'Jayce': {'items': [], 'star': 1}, 'WuKong': {'items': [], 'star': 2}}",platinum,8,season 3,KR-USER-22015,0,1


In [129]:
#전체 시너지 조회
all_traits = set()

for x in df_all_match_2['combination']:
    all_traits.update(x.keys())

all_traits

{'Blaster',
 'Chrono',
 'Cybernetic',
 'DarkStar',
 'Demolitionist',
 'Infiltrator',
 'ManaReaver',
 'MechPilot',
 'Mercenary',
 'Protector',
 'Rebel',
 'Set3_Blademaster',
 'Set3_Brawler',
 'Set3_Celestial',
 'Set3_Mystic',
 'Set3_Sorcerer',
 'Set3_Void',
 'Sniper',
 'SpacePirate',
 'StarGuardian',
 'Starship',
 'Valkyrie',
 'Vanguard'}

In [130]:
#전체 챔피언 조회
all_champion = set()

for x in df_all_match_2['champion']:
    all_champion.update(x.keys())

all_champion


{'Ahri',
 'Annie',
 'Ashe',
 'AurelionSol',
 'Blitzcrank',
 'Caitlyn',
 'ChoGath',
 'Darius',
 'Ekko',
 'Ezreal',
 'Fiora',
 'Fizz',
 'Gangplank',
 'Graves',
 'Irelia',
 'JarvanIV',
 'Jayce',
 'Jhin',
 'Jinx',
 'KaiSa',
 'Karma',
 'Kassadin',
 'Kayle',
 'KhaZix',
 'Leona',
 'Lucian',
 'Lulu',
 'Lux',
 'Malphite',
 'MasterYi',
 'MissFortune',
 'Mordekaiser',
 'Neeko',
 'Poppy',
 'Rakan',
 'Rumble',
 'Shaco',
 'Shen',
 'Sona',
 'Soraka',
 'Syndra',
 'Thresh',
 'TwistedFate',
 'VelKoz',
 'Vi',
 'WuKong',
 'Xayah',
 'Xerath',
 'XinZhao',
 'Yasuo',
 'Ziggs',
 'Zoe'}

In [131]:
champion_traits = {
    'Ahri': ['StarGuardian', 'Set3_Sorcerer'],
    'Annie': ['MechPilot', 'Set3_Sorcerer'],
    'Ashe': ['Set3_Celestial', 'Sniper'],
    'AurelionSol': ['Rebel', 'Starship'],
    'Blitzcrank': ['Chrono', 'Set3_Brawler'],
    'Caitlyn': ['Chrono', 'Sniper'],
    'ChoGath': ['Set3_Void', 'Set3_Brawler'],
    'Darius': ['SpacePirate', 'ManaReaver'],
    'Ekko': ['Cybernetic', 'Infiltrator'],
    'Ezreal': ['Chrono', 'Blaster'],
    'Fiora': ['Cybernetic', 'Set3_Blademaster'],
    'Fizz': ['MechPilot', 'Infiltrator'],
    'Gangplank': ['SpacePirate', 'Demolitionist', 'Mercenary'],
    'Graves': ['SpacePirate', 'Blaster'],
    'Irelia': ['Cybernetic', 'Set3_Blademaster', 'ManaReaver'],
    'JarvanIV': ['DarkStar', 'Protector'],
    'Jayce': ['SpacePirate', 'Vanguard'],
    'Jhin': ['DarkStar', 'Sniper'],
    'Jinx': ['Rebel', 'Blaster'],
    'KaiSa': ['Valkyrie', 'Infiltrator'],
    'Karma': ['DarkStar', 'Set3_Mystic'],
    'Kassadin': ['Set3_Celestial', 'ManaReaver'],
    'Kayle': ['Valkyrie', 'Set3_Blademaster'],
    'KhaZix': ['Set3_Void', 'Infiltrator'],
    'Leona': ['Cybernetic', 'Vanguard'],
    'Lucian': ['Cybernetic', 'Blaster'],
    'Lulu': ['Set3_Celestial', 'Set3_Mystic'],
    'Lux': ['DarkStar', 'Set3_Sorcerer'],
    'Malphite': ['Rebel', 'Set3_Brawler'],
    'MasterYi': ['Rebel', 'Set3_Blademaster'],
    'MissFortune': ['Valkyrie', 'Blaster', 'Mercenary'],
    'Mordekaiser': ['DarkStar', 'Vanguard'],
    'Neeko': ['StarGuardian', 'Protector'],
    'Poppy': ['StarGuardian', 'Vanguard'],
    'Rakan': ['Set3_Celestial', 'Protector'],
    'Rumble': ['MechPilot', 'Demolitionist'],
    'Shaco': ['DarkStar', 'Infiltrator'],
    'Shen': ['Chrono', 'Set3_Blademaster'],
    'Sona': ['Rebel', 'Set3_Mystic'],
    'Soraka': ['StarGuardian', 'Set3_Mystic'],
    'Syndra': ['StarGuardian', 'Set3_Sorcerer'],
    'Thresh': ['Chrono', 'ManaReaver'],
    'TwistedFate': ['Chrono', 'Set3_Sorcerer'],
    'VelKoz': ['Set3_Void', 'Set3_Sorcerer'],
    'Vi': ['Cybernetic', 'Set3_Brawler'],
    'WuKong': ['Chrono', 'Vanguard'],
    'Xayah': ['Set3_Celestial', 'Set3_Blademaster'],
    'Xerath': ['DarkStar', 'Set3_Sorcerer'],
    'XinZhao': ['Set3_Celestial', 'Protector'],
    'Yasuo': ['Rebel', 'Set3_Blademaster'],
    'Ziggs': ['Rebel', 'Demolitionist'],
    'Zoe': ['StarGuardian', 'Set3_Sorcerer']
}


In [132]:
#뒤집개템
item_to_trait = {
    18: 'Set3_Blademaster',   # Blade of the Ruined King
    28: 'Infiltrator',        # Infiltrator's Talons
    38: 'Demolitionist',      # Demolitionist's Charge
    48: 'StarGuardian',       # Star Guardian's Charm
    58: 'Rebel',              # Rebel Medal
    68: 'Set3_Celestial',     # Celestial Orb
    78: 'Protector',          # Protector's Chestguard
    89: 'DarkStar'            # Dark Star's Heart
}

In [133]:
# champion 컬럼의 챔피언 + 아이템 정보를 이용해
# 기본 시너지와 아이템 추가 시너지를 함께 세어 combination 형태로 만드는 함수v2

def make_combination_from_champion_and_items(champion_dict):
    # 시너지 카운트 저장용 딕셔너리
    trait_count = {}
    
    # 챔피언 이름, 정보 하나씩 반복
    for champ_name, champ_info in champion_dict.items():
        # 챔피언 기본 시너지 조회
        base_traits = champion_traits.get(champ_name, [])

        # 기본 시너지 개수 누적
        for trait in base_traits:
            trait_count[trait] = trait_count.get(trait, 0) + 1
        
        # 장착 아이템 목록 조회
        item_ids = champ_info.get('items', [])
        
        # 아이템이 추가 시너지를 주는지 확인
        for item_id in item_ids:
            added_trait = item_to_trait.get(item_id)

            # 추가 시너지가 있으면 개수 누적
            if added_trait:
                trait_count[added_trait] = trait_count.get(added_trait, 0) + 1
    
    # 시너지별 개수 반환
    return trait_count

In [134]:
# 혹시 모르니까 revuilt_v2 컬럼 만들기.
df_all_match_2['combination_rebuilt_v2'] = df_all_match_2['combination'].copy()

In [135]:
#채울 대상
mask = (
    (df_all_match_2['champion'] != {}) &
    (df_all_match_2['combination'] == {})
)

#아이템 반영한 함수 실행
df_all_match_2.loc[mask, 'combination_rebuilt_v2'] = (  #여기만 수정
    df_all_match_2.loc[mask, 'champion']
    .apply(make_combination_from_champion_and_items)
)

# 개수 계산
original_empty = (df_all_match_2['combination'] == {}).sum()
rebuilt_empty = (df_all_match_2['combination_rebuilt_v2'] == {}).sum()
filled_count = original_empty - rebuilt_empty
total_rows = len(df_all_match_2)

# 비율 계산
original_empty_ratio = original_empty / total_rows * 100
rebuilt_empty_ratio = rebuilt_empty / total_rows * 100
filled_ratio_total = filled_count / total_rows * 100
filled_ratio_from_empty = filled_count / original_empty * 100 if original_empty != 0 else 0

# 출력
print("=========함수v1 실행 결과=========")
print(f"원본 combination 빈값 개수: {original_empty} ({original_empty_ratio:.2f}%)")
print(f"새 컬럼 combination_rebuilt 빈값 개수: {rebuilt_empty} ({rebuilt_empty_ratio:.2f}%)")
print(f"채운 개수: {filled_count} ({filled_ratio_total:.2f}% / 빈값 기준 {filled_ratio_from_empty:.2f}%)")


=========함수v1 실행 결과=========
원본 combination 빈값 개수: 117 (0.03%)
새 컬럼 combination_rebuilt 빈값 개수: 0 (0.00%)
채운 개수: 117 (0.03% / 빈값 기준 100.00%)


In [136]:
#확인
df_all_match_2.loc[
    mask,
    ['gameid', 'champion', 'combination', 'combination_rebuilt_v2']
].head(10)

,gameid,champion,combination,combination_rebuilt_v2
6522,KR_4271274609,"{'Fiora': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 2}, 'KaiSa': {'items': [], 'star': 1}, 'Mordekaiser': {'items': [], 'star': 2}, 'Kassadin': {'items': [], 'star': 1}}",{},"{'Cybernetic': 2, 'Set3_Blademaster': 1, 'Vanguard': 2, 'Valkyrie': 1, 'Infiltrator': 1, 'DarkStar': 1, 'Set3_Celestial': 1, 'ManaReaver': 1}"
9873,KR_4359182660,"{'JarvanIV': {'items': [], 'star': 1}, 'Lux': {'items': [], 'star': 1}}",{},"{'DarkStar': 2, 'Protector': 1, 'Set3_Sorcerer': 1}"
15744,KR_4320179268,"{'Malphite': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 1}, 'Mordekaiser': {'items': [], 'star': 1}, 'Ashe': {'items': [], 'star': 1}}",{},"{'Rebel': 1, 'Set3_Brawler': 1, 'Cybernetic': 1, 'Vanguard': 2, 'DarkStar': 1, 'Set3_Celestial': 1, 'Sniper': 1}"
20393,KR_4380989360,"{'Fiora': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [], 'star': 2}, 'Irelia': {'items': [16, 19, 29], 'star': 2}, 'WuKong': {'items': [], 'star': 2}, 'Thresh': {'items': [], 'star': 1}, 'Ekko': {'items': [], 'star': 2}}",{},"{'Cybernetic': 6, 'Set3_Blademaster': 2, 'Vanguard': 2, 'Blaster': 1, 'Set3_Brawler': 1, 'ManaReaver': 2, 'Chrono': 2, 'Infiltrator': 1}"
21498,KR_4366152411,"{'Malphite': {'items': [], 'star': 2}, 'KhaZix': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 3}, 'Mordekaiser': {'items': [], 'star': 2}, 'Shaco': {'items': [], 'star': 2}, 'Jayce': {'items': [], 'star': 1}, 'WuKong': {'items': [], 'star': 2}}",{},"{'Rebel': 1, 'Set3_Brawler': 1, 'Set3_Void': 1, 'Infiltrator': 2, 'Cybernetic': 1, 'Vanguard': 4, 'DarkStar': 2, 'SpacePirate': 1, 'Chrono': 1}"
39767,KR_4367362223,"{'Darius': {'items': [], 'star': 2}, 'Rakan': {'items': [], 'star': 2}, 'XinZhao': {'items': [], 'star': 2}, 'Jayce': {'items': [], 'star': 3}, 'Kassadin': {'items': [], 'star': 2}, 'Ashe': {'items': [], 'star': 2}, 'Jhin': {'items': [], 'star': 1}}",{},"{'SpacePirate': 2, 'ManaReaver': 2, 'Set3_Celestial': 4, 'Protector': 2, 'Vanguard': 1, 'Sniper': 2, 'DarkStar': 1}"
40695,KR_4289735897,"{'Malphite': {'items': [], 'star': 2}, 'Fiora': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 2}, 'Yasuo': {'items': [], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [], 'star': 2}, 'Irelia': {'items': [], 'star': 1}, 'AurelionSol': {'items': [], 'star': 2}}",{},"{'Rebel': 3, 'Set3_Brawler': 2, 'Cybernetic': 5, 'Set3_Blademaster': 3, 'Vanguard': 1, 'Blaster': 1, 'ManaReaver': 1, 'Starship': 1}"
44146,KR_4345979831,"{'Malphite': {'items': [], 'star': 2}, 'KhaZix': {'items': [], 'star': 2}, 'Graves': {'items': [], 'star': 2}, 'KaiSa': {'items': [], 'star': 2}, 'Shaco': {'items': [], 'star': 2}}",{},"{'Rebel': 1, 'Set3_Brawler': 1, 'Set3_Void': 1, 'Infiltrator': 3, 'SpacePirate': 1, 'Blaster': 1, 'Valkyrie': 1, 'DarkStar': 1}"
68525,KR_4294692860,"{'TwistedFate': {'items': [], 'star': 2}, 'Malphite': {'items': [], 'star': 2}, 'KhaZix': {'items': [], 'star': 2}, 'Blitzcrank': {'items': [], 'star': 2}, 'Vi': {'items': [], 'star': 1}, 'ChoGath': {'items': [], 'star': 2}, 'VelKoz': {'items': [], 'star': 2}}",{},"{'Chrono': 2, 'Set3_Sorcerer': 2, 'Rebel': 1, 'Set3_Brawler': 4, 'Set3_Void': 3, 'Infiltrator': 1, 'Cybernetic': 1}"
82606,KR_4400073539,"{'KhaZix': {'items': [], 'star': 2}, 'KaiSa': {'items': [], 'star': 2}, 'Annie': {'items': [], 'star': 2}, 'Shaco': {'items': [], 'star': 2}, 'Rumble': {'items': [], 'star': 1}, 'WuKong': {'items': [], 'star': 2}}",{},"{'Set3_Void': 1, 'Infiltrator': 3, 'Valkyrie': 1, 'MechPilot': 2, 'Set3_Sorcerer': 1, 'DarkStar': 1, 'Demolitionist': 1, 'Chrono': 1, 'Vanguard': 1}"


In [137]:
mask_eval = df_all_match_2['combination'] != {}

df_all_match_2.loc[mask_eval, 'combination_rebuilt_v2_test'] = (
    df_all_match_2.loc[mask_eval, 'champion']
    .apply(make_combination_from_champion_and_items)
)

exact_match_ratio = (
    df_all_match_2.loc[mask_eval, 'combination'] ==
    df_all_match_2.loc[mask_eval, 'combination_rebuilt_v2_test']
).mean()

print(exact_match_ratio)

0.98798516125776


In [138]:
test = df_all_match_2['combination'] != {}
test.value_counts()

combination
True     396260
False       117
Name: count, dtype: int64

In [139]:
test = df_all_match_2['combination_rebuilt_v2'] != {}
test.value_counts()

combination_rebuilt_v2
True    396377
Name: count, dtype: int64

In [140]:
df_all_match_2['combination'] = df_all_match_2['combination_rebuilt_v2']

In [141]:
test = df_all_match_2['combination'] != {}
test.value_counts()

combination
True    396377
Name: count, dtype: int64

In [142]:
df_all_match_2 = df_all_match_2.drop(columns='combination_rebuilt_v2')

In [143]:
# 확인
print(f"shape: {df_all_match_2.shape}")
print(f"combination {{}} 개수: {(df_all_match_2['combination'] == '{}').sum()}")

shape: (396377, 15)
combination {} 개수: 0


In [144]:
df_all_match_2.groupby(['gameid', 'tier']).size().value_counts()

8    49363
7      206
1        5
6        3
2        2
4        1
Name: count, dtype: int64

In [145]:
bad_game_ids = ['KR_4271274609', 'KR_4275570139', 'KR_4280509972',
                'KR_4280775299', 'KR_4282545658', 'KR_4289338197',
                'KR_4289735897', 'KR_4290691308', 'KR_4294692860', 
                'KR_4296351103', 'KR_4297130798', 'KR_4297845328', 
                'KR_4299048416', 'KR_4299905690', 'KR_4300306742', 
                'KR_4302185296', 'KR_4303978194', 'KR_4304139091', 
                'KR_4305607733', 'KR_4310913023', 'KR_4311960180', 
                'KR_4312432447', 'KR_4313489084', 'KR_4316731107', 
                'KR_4316871401', 'KR_4317019936', 'KR_4318795367', 
                'KR_4319063178', 'KR_4319453241', 'KR_4320179268', 
                'KR_4320583924', 'KR_4320825079', 'KR_4321051966', 
                'KR_4321916992', 'KR_4323040528', 'KR_4324633175', 
                'KR_4326077114', 'KR_4328925229', 'KR_4328956024', 
                'KR_4329179072', 'KR_4329469775', 'KR_4329943985', 
                'KR_4329954266', 'KR_4331830791', 'KR_4332575237', 
                'KR_4333355139', 'KR_4333774624', 'KR_4334218028', 
                'KR_4334932041', 'KR_4335172878', 'KR_4335428064', 
                'KR_4336530978', 'KR_4336652699', 'KR_4337925947', 
                'KR_4338428849', 'KR_4338759111', 'KR_4339224641', 
                'KR_4339747470', 'KR_4339755956', 'KR_4339954211', 
                'KR_4342964452', 'KR_4343037266', 'KR_4343803997', 
                'KR_4344420157', 'KR_4345849647', 'KR_4345979831', 
                'KR_4346725443', 'KR_4347420668', 'KR_4347563312', 
                'KR_4348553248', 'KR_4348695357', 'KR_4348875448', 
                'KR_4348978118', 'KR_4350840186', 'KR_4351944962', 
                'KR_4355031855', 'KR_4355315253', 'KR_4355789764', 
                'KR_4356006193', 'KR_4356103166', 'KR_4357766705', 
                'KR_4357940487', 'KR_4357946771', 'KR_4358516973', 
                'KR_4358979405', 'KR_4359182660', 'KR_4359229731', 
                'KR_4359443422', 'KR_4359918919', 'KR_4360480116', 
                'KR_4360487941', 'KR_4360680648', 'KR_4360907133', 
                'KR_4361183625', 'KR_4362168418', 'KR_4362287668', 
                'KR_4362407195', 'KR_4362522169', 'KR_4362564988', 
                'KR_4363251678', 'KR_4363393379', 'KR_4363878896', 
                'KR_4363908926', 'KR_4364923061', 'KR_4365043969', 
                'KR_4365136267', 'KR_4365400496', 'KR_4366152411', 
                'KR_4367362223', 'KR_4371241035', 'KR_4380809484', 
                'KR_4380989360', 'KR_4383383173', 'KR_4392159078', 
                'KR_4400073539'
    
]

df_all_match_2 = df_all_match_2[
    ~df_all_match_2['gameid'].isin(bad_game_ids)
].reset_index(drop=True)

# 확인
print(f"제거 후 shape: {df_all_match_2.shape}")
print(f"combination {{}} 개수: {(df_all_match_2['combination'] == '{}').sum()}")
print(f"champion {{}} 개수: {(df_all_match_2['champion'] == '{}').sum()}")

제거 후 shape: (395457, 15)
combination {} 개수: 0
champion {} 개수: 0


In [146]:
game_counts = df_all_match_2.groupby(['gameid', 'tier']).size()
bad = game_counts[game_counts != 8]
print(bad)

gameid         tier    
KR_4229627884  platinum    7
KR_4240765657  platinum    7
KR_4241537045  diamond     7
KR_4242139521  platinum    7
KR_4247293231  platinum    7
                          ..
KR_4401791157  diamond     7
KR_4402068368  diamond     7
KR_4402070898  diamond     7
KR_4402156615  diamond     7
KR_4402268731  diamond     7
Length: 217, dtype: int64


In [147]:
df_all_match_2.groupby(['gameid', 'tier']).size().value_counts()

8    49248
7      206
1        5
6        3
2        2
4        1
Name: count, dtype: int64

# 티어 순서 정의

In [148]:
# 티어 순서 정의 (숫자가 작을수록 높은 티어)
tier_order = {
    'Challenger': 1,
    'GrandMaster': 2,
    'Master': 3,
    'Diamond': 4,
    'Platinum': 5
}
# tier_rank 컬럼 추가
df_all_match_2['tier_rank'] = df_all_match_2['tier'].map(tier_order)

---
## 4. 티어믹스매치에서 상위 계급의 매치로 취급한 버전

### 4-1. 티어믹스매치에서 하위계급의 데이터 삭제

In [149]:
# 원본 복사
df_upper = df_all_match_2.copy()


# 상위 티어 우선 버전
df_upper = (
    df_upper
    .sort_values('tier_rank')               # 오름차순 → Challenger 먼저
    .drop_duplicates(subset=['gameid', 'ranked'], keep='first')
    .drop(columns='tier_rank')
    .reset_index(drop=True)
)

# 확인
print(f"상위 티어 버전: {df_upper.shape}")

상위 티어 버전: (395305, 15)


---
## 5. 티어믹스매치에서 하위 계급의 매치로 취급한 버전

### 5-1. 티어믹스매치에서 상위계급의 데이터 삭제

In [150]:
# 원본 복사
df_lower = df_all_match_2.copy()

# 하위 티어 우선 버전
df_lower = (
    df_lower
    .sort_values('tier_rank', ascending=False)  # 내림차순 → Platinum 먼저
    .drop_duplicates(subset=['gameid', 'ranked'], keep='first')
    .drop(columns='tier_rank')
    .reset_index(drop=True)
)
print(f"하위 티어 버전: {df_lower.shape}")

하위 티어 버전: (395305, 15)


In [151]:
df_upper.groupby(['gameid', 'tier']).size().value_counts()

8    49229
7      206
1        5
6        3
2        2
4        1
Name: count, dtype: int64

In [152]:
# Infiltrato → Infiltrator 오타 수정
df_champion_info['class'] = df_champion_info['class'].str.replace("'Infiltrato'", "'Infiltrator'")

# 확인
df_champion_info[df_champion_info['class'].str.contains('Infiltrat')]

,name,cost,health,defense,attack,attack_range,speed_of_attack,dps,skill_name,skill_cost,origin,class
18,shaco,3,650,25,70,1,0.80,56,shaco_deceive,30/80,Dark Star,['Infiltrator']
30,ekko,5,850,30,60,1,0.90,54,ekko_chronobreak,50/150,Cybernetic,['Infiltrator']
45,kaisa,2,550,20,50,2,0.75,38,kaisa_missilerain,0/60,Valkyrie,['Infiltrator']
46,khazix,1,500,25,50,1,0.70,35,khazix_tastetheirfear,0/65,Void,['Infiltrator']
51,fizz,4,600,25,60,1,0.80,48,fizz_chumthewaters,80/150,Mech-Pilot,['Infiltrator']
